<a href="https://colab.research.google.com/github/maryamabdelkader/Breast-Cancer-Classification/blob/main/Ecommerce_Sales_Analytics_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-commerce Sales & Customer Analytics

**Portfolio project for a General Data Analyst role**

This project uses the real **Online Retail II** transaction dataset. The goal is to demonstrate a full analyst workflow:

**business questions → data audit → cleaning → SQL → Python analysis → customer segmentation → return analysis → business recommendations**

> **Important:** This notebook intentionally preserves operational anomalies and documents the logic used to handle them. It does not delete unusual observations simply because they look inconvenient.


## 1. Business problem

Imagine an online retailer has given us two years of transactional data and wants to understand:

1. How sales performance changes over time.
2. Which countries and products drive revenue.
3. How dependent the business is on repeat customers.
4. Which customers are most valuable or at risk.
5. Which products show unusual return activity.
6. What actions management should consider.

The project is deliberately **not** focused on machine learning. It is designed to demonstrate practical Data Analyst skills: SQL, Python, data cleaning, KPI analysis, customer analytics, visualization, and business interpretation.


## 2. Dataset and provenance

Source: UCI Machine Learning Repository — **Online Retail II**

https://archive.ics.uci.edu/dataset/502/online+retail+ii

The workbook contains two sheets:

- `Year 2009-2010`
- `Year 2010-2011`

The notebook is designed to run in Google Colab. At the start of the notebook, upload `online_retail_II.xlsx` from your computer; the notebook will use the uploaded file automatically. The raw workbook should not be committed to GitHub.


In [1]:
# Imports used throughout the project.
# Pandas is used for data wrangling, NumPy for numerical operations,
# Matplotlib for visualizations, and sqlite3 for SQL validation examples.

import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Google Colab is used to upload the raw Excel workbook from your computer.
try:
    from google.colab import files
except ImportError:
    files = None

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [2]:
# Upload and locate the raw workbook.
# This project is being run in Google Colab, so the safest approach is to
# upload the Excel workbook directly from your computer at runtime.

if files is not None:
    uploaded = files.upload()

    excel_files = [
        filename
        for filename in uploaded.keys()
        if filename.lower().endswith(".xlsx")
    ]

    if not excel_files:
        raise FileNotFoundError(
            "No .xlsx file was uploaded. Please upload online_retail_II.xlsx."
        )

    # Prefer the expected filename when it is present.
    if "online_retail_II.xlsx" in excel_files:
        DATA_PATH = "online_retail_II.xlsx"
    else:
        DATA_PATH = excel_files[0]

else:
    # Fallback for local Jupyter execution outside Google Colab.
    candidate_paths = [
        "online_retail_II.xlsx",
        os.path.join(os.getcwd(), "online_retail_II.xlsx"),
        os.path.join(os.getcwd(), "..", "online_retail_II.xlsx"),
    ]

    DATA_PATH = next(
        (p for p in candidate_paths if os.path.exists(p)),
        None
    )

    if DATA_PATH is None:
        raise FileNotFoundError(
            "online_retail_II.xlsx was not found. In Google Colab, run the upload cell and select the workbook."
        )

print(f"Dataset found: {DATA_PATH}")
print(f"File size: {os.path.getsize(DATA_PATH) / (1024**2):.2f} MB")


KeyboardInterrupt: 

In [ ]:
# Load both workbook sheets and stack them into one transaction table.
# Keeping the two sheets separate initially is useful for verifying that the merge
# did not change the row counts.

xls = pd.ExcelFile(DATA_PATH)

expected_sheets = ["Year 2009-2010", "Year 2010-2011"]
missing_sheets = [s for s in expected_sheets if s not in xls.sheet_names]
if missing_sheets:
    raise ValueError(f"Expected sheets are missing: {missing_sheets}")

raw_parts = {sheet: pd.read_excel(DATA_PATH, sheet_name=sheet) for sheet in expected_sheets}

for sheet_name, part in raw_parts.items():
    print(f"{sheet_name}: {part.shape[0]:,} rows × {part.shape[1]} columns")

raw = pd.concat(raw_parts.values(), ignore_index=True)

# Clean column-name whitespace only. We do not change values yet.
raw.columns = [c.strip() for c in raw.columns]

raw.head()


## 3. Raw data audit

Before cleaning, we need to understand the structure and identify operational issues.

The important fields are:

| Field | Meaning |
|---|---|
| Invoice | Invoice/order identifier. IDs starting with `C` are cancellations. |
| StockCode | Product or operational code. |
| Description | Product description. |
| Quantity | Units on the line; negative values indicate returns/corrections. |
| InvoiceDate | Transaction date/time. |
| Price | Unit price. |
| Customer ID | Customer identifier; some transactions have no customer ID. |
| Country | Customer country. |


In [ ]:
# Standardize data types before calculating quality statistics.
# Converting Customer ID to numeric allows missing IDs to be handled consistently.

raw["Invoice"] = raw["Invoice"].astype(str).str.strip()
raw["StockCode"] = raw["StockCode"].astype(str).str.strip().str.upper()
raw["Description"] = raw["Description"].astype("string").str.strip()
raw["Country"] = raw["Country"].astype("string").str.strip()
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"], errors="coerce")
raw["Customer ID"] = pd.to_numeric(raw["Customer ID"], errors="coerce").astype("Int64")

# Cancellation invoices are identified by the documented "C" prefix.
raw["IsCancellation"] = raw["Invoice"].str.startswith("C")

audit = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "unique invoices",
        "unique products",
        "unique customers",
        "countries",
        "missing customer IDs (%)",
        "duplicate rows (%)",
        "negative quantity rows",
        "zero/negative price rows",
        "cancellation invoice rows",
    ],
    "value": [
        len(raw),
        raw.shape[1],
        raw["Invoice"].nunique(),
        raw["StockCode"].nunique(),
        raw["Customer ID"].nunique(),
        raw["Country"].nunique(),
        raw["Customer ID"].isna().mean() * 100,
        raw.duplicated().mean() * 100,
        (raw["Quantity"] < 0).sum(),
        (raw["Price"] <= 0).sum(),
        raw["IsCancellation"].sum(),
    ]
})

audit


In [ ]:
# Inspect the most important anomaly types instead of deleting them immediately.

print("Invoice prefixes:")
print(raw["Invoice"].str[0].value_counts())

print("\nMost common operational/service stock codes:")
print(raw["StockCode"].value_counts().head(20))

print("\nExamples of zero-price records:")
display(raw.loc[raw["Price"] <= 0,
                ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]]
          .head(20))

print("\nExamples of cancellations/negative quantities:")
display(raw.loc[raw["Quantity"] < 0,
                ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]]
          .head(20))


## 4. Cleaning strategy

The same transaction table can support different analytical questions, so we create explicit analytical views.

### A. Product lines

Keep records with a positive price and exclude known operational/service codes. This gives us a consistent product-line universe.

### B. Gross sales

For sales performance, keep:

- non-cancellation invoices
- positive quantity
- positive price
- product lines only

Revenue = `Quantity × Price`

### C. Returns

Keep negative-quantity product lines separately. This allows us to measure returns rather than silently removing them.

### D. Customer analytics

Customer-level analysis uses records with a non-missing Customer ID. We do **not** discard missing-ID transactions from overall sales analysis because those transactions can still be legitimate sales.


In [ ]:
# These codes were identified during inspection as operational/service items rather than
# ordinary merchandise. They are excluded from product-sales analysis.
#
# We use an explicit list rather than a fragile rule such as "stock code must be numeric",
# because many legitimate products have alphanumeric stock codes (for example 85123A).

NON_PRODUCT_CODES = {
    "POST", "DOT", "M", "D", "C2", "S",
    "BANK CHARGES", "AMAZONFEE", "CRUK",
    "PADS", "B", "TEST001", "ADJUSTMENT"
}

raw["IsNonProduct"] = raw["StockCode"].isin(NON_PRODUCT_CODES)

# Product universe: priced product lines only.
product = raw[
    (~raw["IsNonProduct"])
    & raw["Price"].notna()
    & (raw["Price"] > 0)
].copy()

product["Revenue"] = product["Quantity"] * product["Price"]

# Gross sales universe: positive merchandise sales only.
sales = product[
    (~product["IsCancellation"])
    & (product["Quantity"] > 0)
].copy()

# Returns/corrections: negative product quantities.
returns = product[product["Quantity"] < 0].copy()

# Identified-customer sales, used only where customer-level attribution is required.
customer_sales = sales[sales["Customer ID"].notna()].copy()

print(f"Raw rows:                 {len(raw):,}")
print(f"Product-line rows:        {len(product):,}")
print(f"Gross-sales rows:         {len(sales):,}")
print(f"Return rows:              {len(returns):,}")
print(f"Customer-attributed rows: {len(customer_sales):,}")


In [ ]:
# Calculate core KPIs from the cleaned analytical views.

kpis = {
    "Raw rows": len(raw),
    "Raw unique invoices": raw["Invoice"].nunique(),
    "Raw unique customers": raw["Customer ID"].nunique(),
    "Raw unique stock codes": raw["StockCode"].nunique(),
    "Raw countries": raw["Country"].nunique(),
    "Missing customer IDs (%)": raw["Customer ID"].isna().mean() * 100,
    "Duplicate rows (%)": raw.duplicated().mean() * 100,
    "Gross sales (£)": sales["Revenue"].sum(),
    "Returns value (£)": -returns["Revenue"].sum(),
    "Net product revenue (£)": product["Revenue"].sum(),
    "Gross-sales orders": sales["Invoice"].nunique(),
    "Identified customers": customer_sales["Customer ID"].nunique(),
}

kpi_table = pd.DataFrame({"KPI": kpis.keys(), "Value": kpis.values()})
kpi_table


## 5. Time-series sales analysis

We measure monthly gross sales, units, orders, customers, and average order value.

Gross sales are used here because management usually wants to see demand generation separately from return activity. Net revenue is analyzed later.


In [ ]:
monthly = (
    sales.assign(Month=sales["InvoiceDate"].dt.to_period("M").astype(str))
         .groupby("Month")
         .agg(
             revenue=("Revenue", "sum"),
             units=("Quantity", "sum"),
             orders=("Invoice", "nunique"),
             customers=("Customer ID", "nunique"),
         )
         .reset_index()
)

monthly["aov"] = monthly["revenue"] / monthly["orders"]
monthly["revenue_mom_pct"] = monthly["revenue"].pct_change() * 100

display(monthly)

# Sanity check: the monthly values must sum back to gross sales.
assert np.isclose(monthly["revenue"].sum(), sales["Revenue"].sum())


In [ ]:
# Visualize the monthly gross revenue trend.

plt.figure(figsize=(10, 5))
plt.plot(pd.to_datetime(monthly["Month"]), monthly["revenue"], marker="o")
plt.title("Monthly Gross Sales Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Geographic analysis

We compare countries using revenue, orders, customers, and average order value.

A high average order value can be useful, but it must be interpreted alongside the number of orders. A country with a single large wholesale order should not automatically be treated as the strongest market.


In [ ]:
country = (
    sales.groupby("Country")
         .agg(
             revenue=("Revenue", "sum"),
             orders=("Invoice", "nunique"),
             units=("Quantity", "sum"),
             customers=("Customer ID", "nunique"),
         )
         .assign(aov=lambda x: x["revenue"] / x["orders"])
         .sort_values("revenue", ascending=False)
         .reset_index()
)

country["revenue_share_pct"] = country["revenue"] / country["revenue"].sum() * 100

display(country.head(15))

# Sanity check: country totals must reconcile to gross sales.
assert np.isclose(country["revenue"].sum(), sales["Revenue"].sum())


In [ ]:
top10 = country.head(10).sort_values("revenue")

plt.figure(figsize=(9, 5))
plt.barh(top10["Country"], top10["revenue"])
plt.title("Top 10 Countries by Gross Sales Revenue")
plt.xlabel("Revenue (£)")
plt.tight_layout()
plt.show()


## 7. Product performance

We rank products by revenue and also report units and order reach.

This prevents us from confusing "most units sold" with "most revenue generated".


In [ ]:
product_performance = (
    sales.groupby(["StockCode", "Description"], dropna=False)
         .agg(
             revenue=("Revenue", "sum"),
             units=("Quantity", "sum"),
             orders=("Invoice", "nunique"),
         )
         .assign(avg_unit_price=lambda x: x["revenue"] / x["units"])
         .sort_values("revenue", ascending=False)
         .reset_index()
)

display(product_performance.head(20))

# The first product should equal the first row after sorting by revenue.
assert product_performance.iloc[0]["revenue"] >= product_performance.iloc[1]["revenue"]


In [ ]:
top_products = product_performance.head(10).sort_values("revenue")
labels = top_products["Description"].fillna(top_products["StockCode"]).str.slice(0, 35)

plt.figure(figsize=(10, 6))
plt.barh(labels, top_products["revenue"])
plt.title("Top 10 Products by Gross Sales Revenue")
plt.xlabel("Revenue (£)")
plt.tight_layout()
plt.show()


## 8. Customer analysis and RFM segmentation

We only use transactions with an identified Customer ID here.

RFM means:

- **Recency**: how recently the customer purchased.
- **Frequency**: how many distinct orders they placed.
- **Monetary**: how much revenue they generated.

The snapshot date is one day after the latest known sales date, so smaller Recency values mean more recent activity.


In [ ]:
snapshot_date = customer_sales["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    customer_sales.groupby("Customer ID")
                  .agg(
                      RecencyDays=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
                      Frequency=("Invoice", "nunique"),
                      Monetary=("Revenue", "sum"),
                  )
                  .reset_index()
)

# Rank before qcut so ties in the raw data cannot make quantile bins fail.
rfm["R_score"] = pd.qcut(
    rfm["RecencyDays"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["RFM_Score"] = (
    rfm["R_score"] * 100
    + rfm["F_score"] * 10
    + rfm["M_score"]
)

def assign_segment(row):
    # These rules create interpretable business groups rather than relying
    # on a black-box clustering algorithm.
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Champions"
    if row["R_score"] >= 4 and row["F_score"] >= 3:
        return "Loyal / Active"
    if row["R_score"] >= 4 and row["M_score"] >= 4:
        return "High-Value New"
    if row["R_score"] <= 2 and row["F_score"] >= 4:
        return "At Risk"
    if row["R_score"] <= 2 and row["M_score"] >= 4:
        return "High-Value At Risk"
    if row["R_score"] <= 2 and row["F_score"] <= 2:
        return "Hibernating"
    return "Potential Loyalist"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

rfm.head()


In [ ]:
# Summarize customer segments.

segments = (
    rfm["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .reset_index(name="Customers")
)

segments["Customer Share (%)"] = segments["Customers"] / len(rfm) * 100

display(segments)

plt.figure(figsize=(9, 5))
plot_segments = segments.sort_values("Customers")
plt.barh(plot_segments["Segment"], plot_segments["Customers"])
plt.title("Customer Segments (RFM)")
plt.xlabel("Customers")
plt.tight_layout()
plt.show()


## 9. Repeat customer analysis

A useful business question is whether revenue depends mostly on one-time buyers or returning customers.

We define a repeat customer as someone with **at least two distinct invoices** in the identified-customer dataset.


In [ ]:
orders_per_customer = (
    customer_sales.groupby("Customer ID")["Invoice"]
                  .nunique()
)

repeat_customer_pct = orders_per_customer.ge(2).mean() * 100

customer_metrics = pd.DataFrame({
    "Metric": [
        "Identified customers",
        "Repeat customers (2+ orders)",
        "Repeat customer share (%)",
        "Median orders per customer",
        "Average orders per customer",
    ],
    "Value": [
        orders_per_customer.size,
        int(orders_per_customer.ge(2).sum()),
        repeat_customer_pct,
        orders_per_customer.median(),
        orders_per_customer.mean(),
    ]
})

customer_metrics


## 10. Return analysis

Returns are measured from negative-quantity product lines.

Return rate by SKU is:

`returned units ÷ sold units`

Some products can show extremely high return rates because the raw operational data contains bulk cancellations/corrections. Those records are useful signals for investigation, but they should not automatically be interpreted as evidence of a defective product.


In [ ]:
sold_units = sales.groupby("StockCode")["Quantity"].sum()
returned_units = -returns.groupby("StockCode")["Quantity"].sum()

return_by_product = pd.DataFrame({
    "sold_units": sold_units,
    "returned_units": returned_units,
}).fillna(0)

return_by_product["return_rate_pct"] = (
    return_by_product["returned_units"] / return_by_product["sold_units"] * 100
)

return_by_product = (
    return_by_product
    .join(sales.groupby("StockCode")["Description"].first(), how="left")
    .reset_index()
)

return_by_product = return_by_product[return_by_product["sold_units"] > 0]

display(
    return_by_product
    .sort_values(["returned_units", "return_rate_pct"], ascending=False)
    .head(20)
)

overall_return_value = -returns["Revenue"].sum()
overall_net_revenue = product["Revenue"].sum()

print(f"Gross sales value: £{sales['Revenue'].sum():,.2f}")
print(f"Return value:      £{overall_return_value:,.2f}")
print(f"Net revenue:       £{overall_net_revenue:,.2f}")

# Reconciliation check.
assert np.isclose(
    sales["Revenue"].sum() + returns["Revenue"].sum(),
    product["Revenue"].sum()
)


In [ ]:
# Compare gross sales with net revenue after returns.

monthly_net = (
    product.assign(Month=product["InvoiceDate"].dt.to_period("M").astype(str))
           .groupby("Month")["Revenue"]
           .sum()
           .reset_index()
)

plt.figure(figsize=(10, 5))
plt.plot(pd.to_datetime(monthly_net["Month"]), monthly_net["Revenue"], marker="o")
plt.axhline(0)
plt.title("Monthly Net Revenue After Returns")
plt.xlabel("Month")
plt.ylabel("Net revenue (£)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 11. SQL validation and business queries

The standalone file `sql/business_questions.sql` contains reusable SQLite queries.

For a production portfolio, the important point is not just writing SQL but writing **business-oriented SQL**: monthly KPIs, country performance, product rankings, repeat customers, and returns.

A full 1M+ row SQLite load is intentionally kept out of the default notebook execution to avoid making the notebook unnecessarily slow on modest laptops. The same analytical logic is available in the SQL file.


In [ ]:
# Example SQL query written against the cleaned_sales table.
# The SQL file in /sql/business_questions.sql contains the full query set.
#
# To use it, create a cleaned_sales table in SQLite with the columns documented
# in that SQL file, then execute:

example_sql = """
SELECT
    strftime('%Y-%m', InvoiceDate) AS month,
    ROUND(SUM(Revenue), 2) AS gross_revenue,
    COUNT(DISTINCT Invoice) AS orders,
    SUM(Quantity) AS units_sold
FROM cleaned_sales
WHERE IsCancellation = 0
  AND Quantity > 0
GROUP BY month
ORDER BY month;
"""

print(example_sql)


## 12. Business findings

The verified analysis shows several portfolio-worthy findings:

### Sales
- Cleaned gross merchandise sales are approximately **£20.12M**.
- November 2011 is the strongest month by gross revenue at approximately **£1.46M**.

### Geography
- The **United Kingdom contributes about 85.7%** of cleaned gross sales revenue, making the business highly concentrated in its home market.
- The next largest markets are much smaller, so international expansion should be evaluated carefully rather than assumed to be equally important everywhere.

### Customers
- There are **5,861 identified customers** in the cleaned positive-sales data.
- About **72.3%** of identified customers made at least two distinct purchases, indicating a substantial repeat-customer base.

### Returns
- Return/correction lines represent approximately **£0.73M** of value.
- A few SKUs show extremely high return rates because operational cancellation/correction activity can be concentrated in individual products. These require transaction-level investigation before operational conclusions are made.

### Portfolio lesson
The most important analyst skill demonstrated here is not the charting itself: it is connecting messy transactional records to business definitions and explicitly validating the resulting KPIs.


## 13. Business recommendations

1. **Protect the repeat-customer base.** With roughly three quarters of identified customers purchasing more than once, retention activity is commercially important.
2. **Investigate high-return SKUs.** Prioritize SKUs with high returned units and unusually high return rates, then inspect the underlying invoices before labeling them as quality problems.
3. **Use seasonality in planning.** Revenue spikes around the final quarter indicate the value of inventory and campaign planning ahead of peak months.
4. **Diversify carefully.** The UK dominates revenue, so international markets should be compared using both revenue and sustainable customer/order depth rather than AOV alone.
5. **Use customer segments operationally.** Champions can support loyalty/upsell programs, while High-Value At Risk customers deserve targeted reactivation.


In [ ]:
# Final automated checks.
# These assertions catch accidental changes to the analytical definitions while
# the notebook is being edited.

assert len(raw) == 1_067_371
assert np.isclose(sales["Revenue"].sum(), 20_121_567.15, rtol=0, atol=0.01)
assert np.isclose(-returns["Revenue"].sum(), 728_651.64, rtol=0, atol=0.01)
assert np.isclose(product["Revenue"].sum(), 19_392_915.51, rtol=0, atol=0.01)
assert customer_sales["Customer ID"].nunique() == 5_861
assert np.isclose(repeat_customer_pct, 72.27435591196041, rtol=0, atol=0.000001)

print("Core reconciliation checks passed.")


## 14. Portfolio takeaway

This project demonstrates a realistic analyst workflow on messy operational data:

**Audit → define business rules → clean → reconcile → analyze → segment → interpret → recommend**

The project is intentionally complementary to the cancer/ML project. Together, they show both technical analytical ability and the ability to answer practical business questions with data.
